# OLIVE FRP: target vs. non-target ERP (Visual Search & SpaceShooter)

The **OLIVE FRP dataset** (`ApocalyVec/olive-physio` on the Hugging Face Hub, or the local
`release/dataset/out/frp_dataset.parquet` shipped with this release) contains per-fixation,
fixation-related-potential (FRP) examples collected across three OLIVE user studies (US1
offline simulation, US2 live deployment, US3 silent target-switch). Each row pairs a
fixation-onset-locked EEG epoch (20 channels x 230 samples, 256 Hz, window `[-0.1, 0.8]` s)
and a pupil epoch with a ground-truth fixated-item label `y` (`1` = target, `0` = non-target),
plus condition, block, and saccade metadata. Full field documentation lives in
`release/dataset/CARD.md`.

OLIVE was evaluated on two task types that share the same fixation-triggered epoch pipeline
but differ substantially in what "target" means perceptually and behaviorally:

- **`visual_search`** -- a classic visual-search task (find-the-target-icon), where a fixation
  on the target item is expected to evoke a centro-parietal P300-like response.
- **`spaceshooter`** -- an in-VR SpaceShooter task, where "target" fixations coincide with a
  much more engaged, motor/attention-demanding game state.

This notebook computes and plots the **target-vs-non-target grand-average ERP**, separately
for each task, at parietal/occipital midline electrodes (Pz, POz, Cz), and reports the mean
target-minus-non-target amplitude in the canonical P300 window (300-500 ms) at Pz. It is meant
as the first example under `release/examples/` -- a template for loading the dataset, handling
its nested EEG array column, and doing a basic condition-averaged ERP analysis.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend; this notebook saves a PNG instead of relying on a GUI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

## Load the dataset

We load the local parquet export. The Hugging Face Hub alternative is shown (commented out)
below -- it returns the identical rows/columns, just via `datasets.load_dataset` instead of
`pandas.read_parquet`.

We only need five columns for this analysis: `subject_id`, `user_study`, `task`, `y`, `eeg`.

In [ ]:
# Try a few likely working directories so this cell runs whether the notebook is executed
# from the repo root or from release/examples/ (its own directory).
_candidates = [
    Path("release/dataset/out/frp_dataset.parquet"),   # cwd == repo root
    Path("../dataset/out/frp_dataset.parquet"),          # cwd == release/examples/
    Path("dataset/out/frp_dataset.parquet"),             # cwd == release/
]
DATASET_PATH = next((p for p in _candidates if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find frp_dataset.parquet next to this notebook. Run release/dataset/export_hf.py "
        "first, or point DATASET_PATH at your local copy."
    )
print(f"Loading {DATASET_PATH.resolve()}")

df = pd.read_parquet(DATASET_PATH)

# --- Alternative: load directly from the Hugging Face Hub ---
# from datasets import load_dataset
# hf_ds = load_dataset("ApocalyVec/olive-physio", split="train")
# df = hf_ds.to_pandas()

df = df[["subject_id", "user_study", "task", "y", "eeg"]]
print(df.shape)
df.head()

## EEG epoch layout and the nested-array column

`eeg` loads from parquet as an **object array of 20 per-channel arrays** (not a clean
`[20, 230]` ndarray), because each channel is stored as a variable-length list-typed column
internally. To get a proper `[20, 230]` float32 tensor for a row, reshape it explicitly:

```python
eeg_2d = np.array(row["eeg"].tolist(), dtype=np.float32)  # -> shape (20, 230)
```

Epoch parameters (from `release/dataset/CARD.md` / `release/dataset/extract_epochs.py`):

- Sampling rate: **256 Hz**
- Epoch window: fixation-onset-locked, `t0 = -0.1` s to `0.8` s -> 230 samples/channel
- Sample index for time `t`: `round((t - (-0.1)) * 256)`
- 20-channel B-Alert X24 subset (10-20 layout). Channel indices used below:
  **Pz = 10**, POz = 16, Cz = 18.

In [ ]:
FS = 256                # Hz
T0 = -0.1                # s, epoch start (fixation onset - 0.1 s)
N_T = 230                 # samples/channel
TIME = T0 + np.arange(N_T) / FS

CH_NAMES = ["Fp1", "F7", "F8", "T4", "T6", "T5", "T3", "Fp2", "O1", "P3",
            "Pz", "F3", "Fz", "F4", "C4", "P4", "POz", "C3", "Cz", "O2"]
PZ, POZ, CZ = 10, 16, 18
assert CH_NAMES[PZ] == "Pz" and CH_NAMES[POZ] == "POz" and CH_NAMES[CZ] == "Cz"


def t_to_idx(t):
    """Time (s) -> sample index within a [-0.1, 0.8] s, 256 Hz epoch."""
    return int(round((t - T0) * FS))


BASELINE_SLICE = slice(t_to_idx(-0.1), t_to_idx(0.0))   # [-0.1, 0] s baseline window
WIN_SLICE = slice(t_to_idx(0.3), t_to_idx(0.5))          # 300-500 ms measurement window


def reshape_eeg(row_val):
    """Nested per-channel object array -> float32 [20, 230] ndarray."""
    return np.array(row_val.tolist(), dtype=np.float32)

## Two-stage grand average (target vs. non-target, per task)

Averaging every epoch together would let subjects/sessions with more trials dominate and would
mix baseline offsets across sessions. Instead we use a **two-stage grand average**:

1. For each `(subject_id, user_study)` session: baseline-correct every epoch on `[-0.1, 0]` s
   (subtract that epoch's own pre-fixation mean, per channel), then average all target
   (`y == 1`) epochs together and all non-target (`y == 0`) epochs together -> one
   per-session target ERP and one per-session non-target ERP.
2. Average those per-session ERPs across sessions -> the grand-average target and
   non-target waveforms.

This is done **separately** for `task == "visual_search"` and `task == "spaceshooter"`, so
session-count / trial-count imbalances between tasks don't bleed into each other.

In [ ]:
def two_stage_grand_average(task_df):
    """Return (grand_target, grand_nontarget), each a [20, 230] float32 array of amplitudes (uV)."""
    per_session_target, per_session_nontarget = [], []
    for (subject_id, user_study), g in task_df.groupby(["subject_id", "user_study"]):
        eeg_stack = np.stack([reshape_eeg(v) for v in g["eeg"].values], axis=0)  # [N, 20, 230]
        baseline = eeg_stack[:, :, BASELINE_SLICE].mean(axis=2, keepdims=True)
        eeg_bc = eeg_stack - baseline
        y = g["y"].values
        if (y == 1).any():
            per_session_target.append(eeg_bc[y == 1].mean(axis=0))
        if (y == 0).any():
            per_session_nontarget.append(eeg_bc[y == 0].mean(axis=0))
    grand_target = np.stack(per_session_target, axis=0).mean(axis=0)
    grand_nontarget = np.stack(per_session_nontarget, axis=0).mean(axis=0)
    return grand_target, grand_nontarget

In [ ]:
TASKS = [("visual_search", "Visual Search"), ("spaceshooter", "SpaceShooter")]

grand = {}
for task_key, task_label in TASKS:
    task_df = df[df["task"] == task_key]
    n_sessions = task_df.groupby(["subject_id", "user_study"]).ngroups
    grand[task_key] = two_stage_grand_average(task_df)
    print(f"{task_label}: {len(task_df)} epochs, {n_sessions} sessions")

## Plot: target vs. non-target grand-average waveforms

Two panels (Visual Search, SpaceShooter), each showing the target and non-target
grand-average waveform at Pz (solid), with POz and Cz shown lightly for reference. The
300-500 ms window is shaded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
win_means_pz = {}

for ax, (task_key, task_label) in zip(axes, TASKS):
    gt, gn = grand[task_key]

    ax.axvspan(0.3, 0.5, color="grey", alpha=0.15, label="300-500 ms")
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)

    ax.plot(TIME, gt[PZ], color="crimson", lw=2, label="target (Pz)")
    ax.plot(TIME, gn[PZ], color="steelblue", lw=2, label="non-target (Pz)")
    ax.plot(TIME, gt[POZ], color="crimson", ls="--", lw=1, alpha=0.5, label="target (POz)")
    ax.plot(TIME, gn[POZ], color="steelblue", ls="--", lw=1, alpha=0.5, label="non-target (POz)")
    ax.plot(TIME, gt[CZ], color="crimson", ls=":", lw=1, alpha=0.5, label="target (Cz)")
    ax.plot(TIME, gn[CZ], color="steelblue", ls=":", lw=1, alpha=0.5, label="non-target (Cz)")

    ax.set_title(task_label)
    ax.set_xlabel("time from fixation onset (s)")

    diff_pz = gt[PZ] - gn[PZ]
    win_means_pz[task_key] = float(diff_pz[WIN_SLICE].mean())

axes[0].set_ylabel("amplitude (uV)")
axes[0].legend(fontsize=7, loc="upper left")
fig.suptitle("Target vs. non-target grand-average ERP (two-stage, per-session baseline-corrected)")
fig.tight_layout()

out_png = Path("erp_target_vs_nontarget.png")
fig.savefig(out_png, dpi=150)
print(f"saved {out_png.resolve()}")

In [ ]:
for task_key, task_label in TASKS:
    print(f"{task_label}: Pz 300-500 ms target-minus-non-target = {win_means_pz[task_key]:+.2f} uV")

## Caveat: the SpaceShooter difference is not a clean P300

(Recall `y == 1` means *target*.)

The Pz 300-500 ms target-minus-non-target amplitude reproduced above is roughly **+12 uV for
SpaceShooter** and roughly **+0.4 uV for Visual Search**. These two effects are not the same
thing:

- **Visual Search** shows a small, transient, centro-parietal-positive difference in the
  canonical P300 latency range -- consistent with (though not proof of) a genuine P300-like
  target-detection response.
- **SpaceShooter** shows a much larger difference, but it is a **sustained, slow-potential-shaped**
  deflection rather than a clean transient P300 -- it onsets early and stays elevated well
  beyond the 300-500 ms window (inspect the SpaceShooter panel above). This pattern is more
  consistent with a slow cortical potential tied to **task engagement or motor/attentional
  demand** during "target" fixations in the shooter game (which coincide with active
  aiming/shooting) than with a stimulus-locked P300 component. Treat the SpaceShooter number
  as a task-engagement signature, not as evidence of a P300, and do not directly compare its
  magnitude to the Visual Search P300-latency effect as if they were the same construct.